# PathSnap: how the four models were run

This walkthrough shows the **actual recorded five-fold cross-validation experiments** for Logistic Regression, Gaussian Naive Bayes, Decision Tree and Random Forest.

Default mode reads the saved experiment files. It does **not** claim to retrain models. Change `RETRAIN_MODELS` below to rerun selected models on the complete original training data; new outputs go into a separate directory. Random Forest originally took about 50 minutes for five fits, excluding scoring and output overhead.

These are classifier-only results against valid derived reference labels. Two unreliable training trips were excluded. AdaBoost, Bayesian tuning, student-testing and HMM evaluation are outside these four recorded CV experiments. Final deployment model files have not been saved by these CV runs.


In [1]:
from pathlib import Path
import sys
import json
import inspect
from datetime import datetime

# Works when opened from either the repository root or its notebooks directory.
ROOT = next((p for parent in [Path.cwd(), *Path.cwd().parents]
             for p in [parent, parent / 'Ml_hackathon1']
             if (p / 'src/pathsnap/train.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the Ml_hackathon1 repository')
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathsnap import config
from pathsnap.models import build_model
from pathsnap.train import cross_validate, make_folds, write_json
from pathsnap.training_handoff import load_feature_dir

RUN = ROOT / 'outputs/experiments/run_001'
FEATURES = ROOT / 'outputs/features'
RETRAIN_MODELS = set()  # Example: {'logistic_regression'}; leave empty to show recorded results.
RETRAIN_OUTPUT = ROOT / 'outputs/training' / datetime.now().strftime('walkthrough_%Y%m%d_%H%M%S')
metadata = json.loads((RUN / 'run.json').read_text())
SEED, JOBS = metadata['seed'], metadata['jobs']
print('Repository:', ROOT)
print('Recorded run:', RUN)
print('Random seed:', SEED, '| Random Forest workers:', JOBS)
print('Mode:', 'retrain selected models' if RETRAIN_MODELS else 'read recorded results')


Repository: /home/rohan-sai/Desktop/ML_hackathon/Ml_hackathon1
Recorded run: /home/rohan-sai/Desktop/ML_hackathon/Ml_hackathon1/outputs/experiments/run_001
Random seed: 42 | Random Forest workers: 2
Mode: read recorded results


## 1. The input data

Each row represents one GPS observation paired with one candidate road. The label is `1` when that candidate matches the derived reference arc, and `0` otherwise. Only the 22 declared features enter the model; IDs, labels, fold IDs and reference metadata are excluded from features. `label_valid=False` rows are not used for fitting.


In [2]:
schema = json.loads((FEATURES / 'feature_schema.json').read_text())
feature_names = schema['feature_columns']
for partition in ['train', 'student_test']:
    candidates_file = pq.ParquetFile(FEATURES / f'{partition}_candidates.parquet')
    points_file = pq.ParquetFile(FEATURES / f'{partition}_points.parquet')
    print(f'{partition}: {candidates_file.metadata.num_rows:,} candidate rows, '
          f'{points_file.metadata.num_rows:,} GPS observations')
print('\nFeature columns:')
print(', '.join(feature_names))
preview = next(pq.ParquetFile(FEATURES / 'train_candidates.parquet').iter_batches(batch_size=8)).to_pandas()
print('\nA few real candidate rows (preview only):')
print(preview[['trip_id', 'point_idx', 'arc_id', 'dist_m', 'heading_diff_deg', 'label', 'label_valid', 'fold']].to_string(index=False))


train: 2,797,071 candidate rows, 208,417 GPS observations
student_test: 329,247 candidate rows, 19,160 GPS observations

Feature columns:
dist_m, frac_along, arc_len_m, dist_rank, dist_rel_min, n_cand, heading_diff_deg, cos_heading, speed_mps, dt_prev, dt_next, win_mean_dist, win_max_dist, win_min_dist, win_hit_frac, out_deg_to, in_deg_from, has_reverse, to_node_dist_m, from_node_dist_m, along_advance_m, speed_dist_resid

A few real candidate rows (preview only):
 trip_id  point_idx  arc_id    dist_m  heading_diff_deg  label  label_valid  fold
00000001          0   64973  0.722768          1.313567      1         True     4
00000001          0  117639 15.132321        178.755798      0         True     4
00000001          0    2641 32.198444          1.442501      0         True     4
00000001          1   64973  0.277461          0.688910      1         True     4
00000001          1  117639 16.079744        179.241730      0         True     4
00000001          2   64973  1.839765   

## 2. Keeping whole trips together

All four models used the same five existing folds. A validation trip never contributes rows to the corresponding training fold. Fold `-1` marks trips excluded by the upstream reference-quality rule. The split is by trip, not by candidate row or GPS observation.


In [3]:
assignments = pd.read_csv(RUN / 'fold_assignments.csv', dtype={'trip_id': 'string'})
assert assignments.trip_id.is_unique
assert set(assignments.validation_fold) - {-1} == set(range(5))
print('Trips by validation fold (-1 means excluded):')
print(assignments.groupby('validation_fold').size().rename('trip_count').to_string())
print('\nExcluded trips:', assignments.loc[assignments.validation_fold.eq(-1), 'trip_id'].tolist())
for fold in range(5):
    training_trips = set(assignments.loc[assignments.validation_fold.ge(0) & assignments.validation_fold.ne(fold), 'trip_id'])
    validation_trips = set(assignments.loc[assignments.validation_fold.eq(fold), 'trip_id'])
    assert training_trips.isdisjoint(validation_trips)
print('Verified: no training/validation trip overlap in any fold.')


Trips by validation fold (-1 means excluded):
validation_fold
-1     2
 0    14
 1    15
 2    17
 3    16
 4    16

Excluded trips: ['00000033', '00000054']
Verified: no training/validation trip overlap in any fold.


## 3. The actual shared training loop

For each fold, create a fresh pipeline, fit it on training trips, score candidate roads in held-out trips, and evaluate those predictions. Median imputation is learned on the training fold. Logistic Regression and Gaussian Naive Bayes also receive standard scaling learned on that fold.

The positive-class probability scores each candidate. The highest-scoring candidate per observation becomes the predicted road. Top-1 accuracy includes candidate misses as errors for observations with valid references. Macro F1 and log loss are candidate-level metrics.


In [4]:
print(inspect.getsource(cross_validate))


def cross_validate(name, train, folds, seed, jobs, params=None):
    results, predictions = [], []
    for fold, (fit, val, inventory) in enumerate(folds):
        model = build_model(name, seed, jobs, params)
        start = perf_counter()
        model.fit(fit[train.features], fit.label.astype(int))
        training_time = perf_counter() - start
        start = perf_counter()
        scores = score_candidates(model, val, train.features)
        inference_time = perf_counter() - start
        metrics, _ = evaluate(val, inventory, scores)
        print(f"  {name} fold {fold}: Top-1={metrics['top1_accuracy']:.4f}, fit={training_time:.1f}s", flush=True)
        results.append({"fold": fold, **metrics, "training_seconds": training_time, "inference_seconds": inference_time})
        predictions.append(scores.assign(fold=fold))
    table = pd.DataFrame(results)
    summary = {
        key: (float(table[key].mean()) if table[key].notna().any() else None)
        for key in ["top1_accuracy", 

In [5]:
# Helper for the four sections below. Fitting happens only for names explicitly
# added to RETRAIN_MODELS. Recorded CV results are never overwritten.
training_cache = None


def demonstrate(name):
    global training_cache
    pipeline = build_model(name, seed=SEED, jobs=JOBS)
    print('Pipeline configuration:')
    print(pipeline)
    if name in RETRAIN_MODELS:
        if training_cache is None:
            training, _, _ = load_feature_dir(FEATURES)
            folds, rerun_assignments = make_folds(training, SEED)
            assert rerun_assignments.astype(str).equals(assignments.astype(str))
            training_cache = training, folds
        training, folds = training_cache
        print('\nExecuting fresh training and five-fold CV on full data...')
        summary, scores_by_fold, predictions = cross_validate(name, training, folds, SEED, JOBS)
        directory = RETRAIN_OUTPUT / name
        directory.mkdir(parents=True)
        scores_by_fold.to_csv(directory / 'cv_folds.csv', index=False)
        predictions.to_csv(directory / 'cv_candidate_scores.csv', index=False)
        write_json(directory / 'cv_summary.json', summary)
        write_json(directory / 'configuration.json', {'seed': SEED, 'features': training.features,
                                                    'parameters': pipeline.get_params(deep=True)})
        print('New CV output:', directory)
    else:
        directory = RUN / name
        print('\nLoading recorded results; this cell does not fit the model.')
        summary = json.loads((directory / 'cv_summary.json').read_text())
        scores_by_fold = pd.read_csv(directory / 'cv_folds.csv')
    assert len(scores_by_fold) == 5 and set(scores_by_fold.fold) == set(range(5))
    assert np.isclose(scores_by_fold.top1_accuracy.mean(), summary['top1_accuracy'])
    print('\nFive held-out-fold results:')
    print(scores_by_fold[['fold', 'top1_accuracy', 'macro_f1', 'log_loss', 'training_seconds']].round(4).to_string(index=False))
    print(f"\nMean Top-1: {summary['top1_accuracy']:.2%}")
    print(f"Candidate Macro F1: {summary['macro_f1']:.4f} | Log loss: {summary['log_loss']:.4f}")
    print('\nRecorded candidate scores (first six rows):')
    score_file = directory / 'cv_candidate_scores.csv'
    if not score_file.exists():
        score_file = directory / 'cv_candidate_scores.sample.csv'
    print(pd.read_csv(score_file, nrows=6,
                      dtype={'trip_id': 'string', 'observation_id': 'string', 'candidate_id': 'string'}).to_string(index=False))
    return summary


## 4. Logistic Regression

Median imputation → standard scaling → Logistic Regression. Maximum 2,000 solver iterations, default regularisation C=1, seed from the recorded run.


In [6]:
logistic_regression_result = demonstrate('logistic_regression')


Pipeline configuration:
Pipeline(steps=[('imputer',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(max_iter=2000, random_state=42))])

Loading recorded results; this cell does not fit the model.

Five held-out-fold results:
 fold  top1_accuracy  macro_f1  log_loss  training_seconds
    0         0.8552    0.9097    0.0760           15.9642
    1         0.8355    0.8933    0.0838           15.5330
    2         0.8730    0.9153    0.0541           15.8046
    3         0.8972    0.9314    0.0545           15.7672
    4         0.7997    0.8813    0.0626           15.0480

Mean Top-1: 85.21%
Candidate Macro F1: 0.9062 | Log loss: 0.0662

Recorded candidate scores (first six rows):
 trip_id observation_id candidate_id  probability  fold
00000011              0          129 7.295703e-01     0
00000011              0          859 4.049957e-02     0

## 5. Gaussian Naive Bayes

Median imputation → standard scaling → GaussianNB. Default variance smoothing is 1e-9.


In [7]:
gaussian_nb_result = demonstrate('gaussian_nb')


Pipeline configuration:
Pipeline(steps=[('imputer',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('scaler', StandardScaler()), ('classifier', GaussianNB())])

Loading recorded results; this cell does not fit the model.

Five held-out-fold results:
 fold  top1_accuracy  macro_f1  log_loss  training_seconds
    0         0.8091    0.7620    0.9110            5.9858
    1         0.7470    0.7696    1.2327            6.7796
    2         0.8425    0.7799    0.5432            6.0612
    3         0.8497    0.7579    0.9102            6.8354
    4         0.7669    0.7585    0.5495            6.3849

Mean Top-1: 80.30%
Candidate Macro F1: 0.7656 | Log loss: 0.8293

Recorded candidate scores (first six rows):
 trip_id observation_id candidate_id  probability  fold
00000011              0          129 9.999992e-01     0
00000011              0          859 8.051556e-01     0
00000011              0          862 9.969448e-01     0
00000011      

## 6. Decision Tree

Median imputation → DecisionTreeClassifier. The baseline has no maximum depth and uses the recorded random seed.


In [8]:
decision_tree_result = demonstrate('decision_tree')


Pipeline configuration:
Pipeline(steps=[('imputer',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('classifier', DecisionTreeClassifier(random_state=42))])

Loading recorded results; this cell does not fit the model.

Five held-out-fold results:
 fold  top1_accuracy  macro_f1  log_loss  training_seconds
    0         0.7414    0.8728    1.3824           56.0178
    1         0.7346    0.8760    1.3597           53.8216
    2         0.7782    0.9012    0.8500           53.2671
    3         0.8185    0.9142    0.8069           48.8399
    4         0.6996    0.8636    1.0440           45.3187

Mean Top-1: 75.45%
Candidate Macro F1: 0.8855 | Log loss: 1.0886

Recorded candidate scores (first six rows):
 trip_id observation_id candidate_id  probability  fold
00000011              0          129          1.0     0
00000011              0          859          0.0     0
00000011              0          862          0.0     0
00000011         

## 7. Random Forest

Median imputation → RandomForestClassifier with 200 trees, no maximum depth, two workers and the recorded random seed.


In [9]:
random_forest_result = demonstrate('random_forest')


Pipeline configuration:
Pipeline(steps=[('imputer',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('classifier',
                 RandomForestClassifier(n_estimators=200, n_jobs=2,
                                        random_state=42))])

Loading recorded results; this cell does not fit the model.

Five held-out-fold results:
 fold  top1_accuracy  macro_f1  log_loss  training_seconds
    0         0.8510    0.9132    0.0788          620.4136
    1         0.8337    0.9045    0.0790          641.6708
    2         0.8848    0.9344    0.0430          604.9330
    3         0.9042    0.9454    0.0443          557.7338
    4         0.8077    0.8937    0.0580          561.7971

Mean Top-1: 85.63%
Candidate Macro F1: 0.9182 | Log loss: 0.0606

Recorded candidate scores (first six rows):
 trip_id observation_id candidate_id  probability  fold
00000011              0          129        0.740     0
00000011              0          859        

## 8. Compare the four completed models

This table is calculated from the outputs shown above. Top-1 is the arithmetic mean across the same five validation folds. A final model must be selected after the required five-model comparison; this notebook does not treat these four baselines as a completed final submission.


In [10]:
results = {
    'Logistic Regression': logistic_regression_result,
    'Gaussian Naive Bayes': gaussian_nb_result,
    'Decision Tree': decision_tree_result,
    'Random Forest': random_forest_result,
}
comparison = pd.DataFrame.from_dict(results, orient='index')
print(comparison[['top1_accuracy', 'macro_f1', 'log_loss', 'training_seconds']].round(4).to_string())


                      top1_accuracy  macro_f1  log_loss  training_seconds
Logistic Regression          0.8521    0.9062    0.0662           15.6234
Gaussian Naive Bayes         0.8030    0.7656    0.8293            6.4094
Decision Tree                0.7545    0.8855    1.0886           51.4530
Random Forest                0.8563    0.9182    0.0606          597.3097


![Recorded four-model comparison](../outputs/experiments/run_001/four_model_comparison.png)

The whiskers show fold standard deviation, not confidence intervals.


## 9. Original terminal evidence

The following lines come from the original local training log. They document the four baseline runs before the local process was interrupted to move AdaBoost to Kaggle. They are historical outputs, not a new training execution by this notebook.


In [11]:
log = (RUN / 'run_001.log').read_text()
first_four_models = log.split('Cross-validating adaboost')[0]
print(first_four_models)
print('\nRecorded environment (first 15 lines):')
print('\n'.join((RUN / 'environment.txt').read_text().splitlines()[:15]))


Cross-validating logistic_regression
  logistic_regression fold 0: Top-1=0.8552, fit=16.0s
  logistic_regression fold 1: Top-1=0.8355, fit=15.5s
  logistic_regression fold 2: Top-1=0.8730, fit=15.8s
  logistic_regression fold 3: Top-1=0.8972, fit=15.8s
  logistic_regression fold 4: Top-1=0.7997, fit=15.0s
Cross-validating gaussian_nb
  gaussian_nb fold 0: Top-1=0.8091, fit=6.0s
  gaussian_nb fold 1: Top-1=0.7470, fit=6.8s
  gaussian_nb fold 2: Top-1=0.8425, fit=6.1s
  gaussian_nb fold 3: Top-1=0.8497, fit=6.8s
  gaussian_nb fold 4: Top-1=0.7669, fit=6.4s
Cross-validating decision_tree
  decision_tree fold 0: Top-1=0.7414, fit=56.0s
  decision_tree fold 1: Top-1=0.7346, fit=53.8s
  decision_tree fold 2: Top-1=0.7782, fit=53.3s
  decision_tree fold 3: Top-1=0.8185, fit=48.8s
  decision_tree fold 4: Top-1=0.6996, fit=45.3s
Cross-validating random_forest
  random_forest fold 0: Top-1=0.8510, fit=620.4s
  random_forest fold 1: Top-1=0.8337, fit=641.7s
  random_forest fold 2: Top-1=0.8848, f

## 10. Reproduce all four from the terminal

Run the following command from the repository root with the project environment activated. It performs real training in a **new** output directory, runs all four models on the original folds, fits baseline pipelines, and evaluates student-testing. It does not perform five-model selection or Bayesian tuning.

```bash
PYTHONPATH=src python -m pathsnap.train \
  --feature-dir outputs/features \
  --output outputs/training/four_model_reproduction_001 \
  --models logistic_regression gaussian_nb decision_tree random_forest \
  --baseline-only --jobs 2 --seed 42
```

The original four CV experiments did not retain their temporary fitted fold models. The terminal command additionally saves refitted `pipeline.joblib` artifacts; do not describe those new artifacts as already existing until this command completes.
